# 01 — Inspect our ACS employment data

Select the project's `.venv/Scripts/python.exe` as the notebook kernel, then run cells from the top. These cells read the already prepared files; they do not download again.

**What to learn:** who is included, what the label means, which groups are small, and which missing values are structural. Each section ends with a question to consider. Test outcomes stay reserved for final evaluation.

`PWGTP` is a person survey weight: a sampled record represents roughly that many people in population estimates. A fictional weight of 100 means approximately 100 represented people, not 100 duplicate rows. We require positive weights but initially train/evaluate with equal weight per sampled record.

References: [2018 Census dictionary](https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2018.txt), [research design](../proposal.md).

**Processing record:** [data_processing_log.md](../data_processing_log.md) documents every rule, column/code, finding, and pending choice for review.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'proposal.md').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data import FEATURES, INPUTS, CATEGORICAL, group_summary, model_frame
DATA = ROOT / 'data' / 'processed'
AUDIT = ROOT / 'results' / 'data_audit'
manifest = json.loads((DATA / 'manifest.json').read_text())
print('Python:', sys.executable)
print('Project:', ROOT)

Python: E:\UCSD\ece\285\privacy-fairness-tradeoffs\.venv\Scripts\python.exe
Project: E:\UCSD\ece\285\privacy-fairness-tradeoffs


### What this setup cell does

We will inspect one main pandas DataFrame, called `visible`. The setup cell only makes the tools and file locations available.

- `import` makes a library's functions available; it does not load or change our records.
- `pandas` (`pd`) handles tables; `numpy` (`np`) supports numerical checks.
- `Path` locates files. `ROOT` searches the current folder and its parents for our project. `DATA` and `AUDIT` are folder paths.
- `sys.path` lets Python find our project's helper code in `src/data.py`.
- `FEATURES` lists 17 source columns; `INPUTS` lists the 14 that the model uses after ESP is dropped. `group_summary` counts groups; `model_frame` prepares column types.
- `manifest` is a small dictionary describing an earlier preparation run. It contains counts and settings, not the individual records.
- `sparse` is only needed for the optional matrix checks near the end.

**The printed paths confirm which Python environment and project folder are being used.**

## 1. Who is in the study?

The eligibility rule is **age > 16, age < 90, and PWGTP >= 1**. This keeps ages 17–89, including older adults. Split assignment uses a fixed random seed and is independent of labels and protected groups.

PWGTP is the person survey weight. A sampled person with a weight of 100 represents roughly 100 people in population estimates. Requiring PWGTP >= 1 checks eligibility

**Inspect:** How many records did filtering remove? How large is the training set?

In [2]:
display(pd.Series({k: manifest[k] for k in ['raw_rows', 'eligible_rows', 'excluded_rows', 'encoded_dimension']}, name='value').to_frame())
display(pd.Series(manifest['split_counts'], name='records').to_frame())
cohort = pd.read_csv(DATA / 'cohort.csv')
visible = cohort.loc[cohort['split'].ne('test')].copy()
assert visible.AGEP.between(17, 89).all()
assert visible.PWGTP.ge(1).all()
display(visible[[c for c in FEATURES if c in visible.columns] + ['ESR', 'Y', 'split']].head(8))

,value
raw_rows,378817
eligible_rows,302640
excluded_rows,76177
encoded_dimension,77


,records
train,151320
test,60528
calibration,30264
reference,30264
development,30264


,AGEP,SCHL,MAR,SEX,DIS,MIG,CIT,MIL,ANC,NATIVITY,RELP,DEAR,DEYE,DREM,RAC1P,GCL,ESR,Y,split
0,30,14.0,1,1,2,3.0,1,4.0,1,1,16,2,2,2.0,8,2.0,6.0,0,calibration
1,18,14.0,5,2,2,1.0,1,4.0,1,1,17,2,2,2.0,1,NaN,6.0,0,reference
2,69,17.0,1,1,1,1.0,1,2.0,2,1,17,2,2,2.0,9,2.0,6.0,0,train
3,25,1.0,5,1,1,1.0,1,4.0,1,1,17,1,2,1.0,1,NaN,6.0,0,train
5,19,18.0,5,1,2,2.0,5,4.0,1,2,17,2,2,2.0,6,NaN,6.0,0,train
6,21,16.0,5,1,2,1.0,1,4.0,2,1,17,2,2,2.0,1,NaN,1.0,1,calibration
7,65,22.0,5,1,2,1.0,1,4.0,1,1,17,2,2,2.0,1,2.0,1.0,1,development
9,82,14.0,1,1,1,1.0,4,4.0,4,2,16,2,2,1.0,6,2.0,6.0,0,train


### How to read these results

| Result | Meaning | This preparation run |
| --- | --- | --- |
| `raw_rows` | People in the downloaded California file before our filters | 378,817 |
| `eligible_rows` | People retained after the age and positive-weight rules | 302,640 |
| `excluded_rows` | People who failed at least one of those rules | 76,177 (20.1% of raw records) |
| `encoded_dimension` | Numerical input columns per person after encoding | 77 |

`raw_rows = eligible_rows + excluded_rows`. Excluded records were not necessarily errors: many people simply fall outside our study population. These counts do not mean we removed everyone with missing features, or removed retirees.

The split table divides the eligible people by purpose: **reference** fits preprocessing/LFR; **train** teaches the MLP; **development** helps choose settings; **calibration** teaches EO rules; **test** is reserved for the final assessment.

**What the code does:**

1. `pd.Series(...).to_frame()` turns saved counts into a display table.
2. `pd.read_csv(...)` loads `cohort.csv` into one DataFrame named `cohort`. It already contains filtered records, raw feature codes, the target, and split assignments.
3. `.loc[...ne('test')].copy()` creates `visible`, our inspection DataFrame, by selecting non-test rows. It has 242,112 rows. Test labels stay out of the descriptive analysis.
4. `assert` checks the age/weight rules and stops with an error if a rule fails. No output means the check passed.
5. `.head(8)` displays eight example records. `ESR` is the original code, `Y` the binary target, and `split` the assigned purpose. `NaN` denotes a missing value.

**77 columns does not mean 77 people.** Encoding expands the 14 input variables into 1 scaled-age column and 76 category-indicator columns; section 7 explains why.

## 2. What does the target actually mean?

`Y = 1` exactly when `ESR = 1`. Crucially, `Y = 0` includes employed people temporarily away from work, as well as unemployed people and people outside the labor force. It is the Folktables benchmark definition.

**Inspect:** Which ESR categories contribute most negative labels? Why would calling all negatives “unemployed” be inaccurate?

In [3]:
esr_names = {1: 'Civilian employed, at work', 2: 'Civilian employed, away from work',
             3: 'Unemployed', 4: 'Armed forces, at work',
             5: 'Armed forces, away from work', 6: 'Not in labor force'}
target = visible.groupby('ESR').agg(n=('Y', 'size'), label=('Y', 'first'))
target['meaning'] = target.index.map(esr_names)
display(target)
print('Positive-label fraction in non-test records:', round(visible.Y.mean(), 4))
age_bands = pd.cut(visible.AGEP, [16, 24, 44, 64, 89], labels=['17–24', '25–44', '45–64', '65–89'])
display(visible.groupby(age_bands, observed=True).Y.agg(n='size', positive_rate='mean'))

,n,label,meaning
ESR,,,
1.0,137806,1,"Civilian employed, at work"
2.0,3850,0,"Civilian employed, away from work"
3.0,7942,0,Unemployed
4.0,1226,0,"Armed forces, at work"
5.0,3,0,"Armed forces, away from work"
6.0,91285,0,Not in labor force


Positive-label fraction in non-test records: 0.5692


,n,positive_rate
AGEP,,
17–24,31786,0.473385
25–44,79287,0.743363
45–64,80730,0.675387
65–89,50309,0.184778


### What the target tables tell us

In the first table, `n` counts records with each ESR code. `label` shows the shared value of `Y` for that code. `meaning` translates the code into words. The code uses `groupby('ESR')` to form these groups, then `agg(...)` to count records and retrieve their label.

`visible.Y.mean()` is the positive-label fraction because `Y` contains zeros and ones. For example, **0.5692 means approximately 56.92% of the inspected sample has Y=1**. It is a property of the labels, not model accuracy: no model has made predictions here.

`pd.cut(...)` places ages into four bands for display. It does not change the model's age feature or exclude additional people. The second table shows each band's size and positive-label fraction. Differences between bands help us understand the population; they do not establish discrimination.

## 3. What is PWGTP doing?

Positive weights determine eligibility. They are excluded from model features. Our current positive rates are unweighted: they describe this sample, rather than survey-weighted California population estimates.

**Inspect:** Do the weights vary substantially? Distinguish filtering on a weight from using it to weight training or metrics.

In [4]:
display(visible.PWGTP.describe().to_frame())
print('PWGTP in model inputs?', 'PWGTP' in INPUTS)

,PWGTP
count,242112.000000
mean,101.770652
std,73.929204
min,1.000000
25%,58.000000
50%,80.000000
75%,122.000000
max,1598.000000


PWGTP in model inputs? False


### How to read `describe()`

- `count`: records with a non-missing weight.
- `mean`: average weight; `std`: how much weights vary.
- `min`/`max`: smallest/largest weight.
- `25%`, `50%`, `75%`: quartiles; `50%` is the median.

These statistics describe the survey weights, not employment probabilities. A mean weight of 100, for example, would not mean 100% employment. We are not multiplying rows or weighting the current summaries by PWGTP.

The printed **False** confirms that `PWGTP` is absent from the model-input list. The expression `... in INPUTS` tests list membership.

## 4. Which values are missing?

Some Census fields are not applicable to certain people. For example, parents' employment status and grandparent fields have specific applicability rules. We retain these records, drop ESP as agreed, and encode remaining categorical N/A explicitly, instead of deleting every row with an empty field.

**Inspect:** Which fields have the most missingness? Check their applicability in the Census dictionary before proposing an exclusion.

In [5]:
missingness = pd.read_csv(AUDIT / 'missingness.csv')
display(missingness.sort_values('cohort_missing_fraction', ascending=False))
display(model_frame(visible).head())

,feature,raw_missing,cohort_missing,cohort_missing_fraction
5,ESP,305431,298334,0.985772
16,GCL,137905,65347,0.215923
0,AGEP,0,0,0.000000
3,SEX,0,0,0.000000
1,SCHL,10952,0,0.000000
4,DIS,0,0,0.000000
6,MIG,3444,0,0.000000
7,CIT,0,0,0.000000
2,MAR,0,0,0.000000
8,MIL,72558,0,0.000000


,AGEP,SCHL,MAR,DIS,MIG,CIT,MIL,ANC,NATIVITY,RELP,DEAR,DEYE,DREM,GCL
0,30,14,1,2,3,1,4,1,1,16,2,2,2,2
1,18,14,5,2,1,1,4,1,1,17,2,2,2,NOT_APPLICABLE
2,69,17,1,1,1,1,2,2,1,17,2,2,2,2
3,25,1,5,1,1,1,4,1,1,17,1,2,1,NOT_APPLICABLE
5,19,18,5,2,2,5,4,1,2,17,2,2,2,NOT_APPLICABLE


### How to interpret missingness

`pd.read_csv(...)` loads a small audit table created during preparation. `sort_values(...)` puts the highest missing fractions first.

- `raw_missing`: empty values before eligibility filtering.
- `cohort_missing`: empty values among all eligible records.
- `cohort_missing_fraction`: that count divided by eligible records. Multiply by 100 to read a percentage.

In this run, **ESP is missing for about 98.6% of eligible people**, while GCL is missing for about 21.6%. Many entries are structurally not applicable. Deleting every record with a missing ESP would remove almost the entire study population. A zero missing count also does not guarantee that every code is substantively valid; applicability/suppression codes need their own interpretation.

`model_frame(visible).head()` shows how the input columns are represented before one-hot encoding. Categorical missing values become `NOT_APPLICABLE`; category codes become strings so we do not treat them as numerical magnitudes. ESP is now dropped by user decision; the audit table retains its original missingness. This helper returns a new table and preserves `visible`. Its name does not mean a neural network has been trained.

## 5. Who are our protected groups?

The recorded sex codes are 1=male and 2=female. Race codes retain the original Census categories; race does not encode Hispanic ethnicity. We audit sex and race separately and inspect their intersections. Rates below are **label base rates**, not model fairness results.

**Inspect:** Which groups have fewer records? Which have few positive or negative labels? Differences in base rates matter when interpreting later fairness metrics.

In [6]:
race_names = {1:'White alone', 2:'Black alone', 3:'American Indian alone',
              4:'Alaska Native alone', 5:'Other American Indian/Alaska Native category',
              6:'Asian alone', 7:'Native Hawaiian/Other Pacific Islander alone',
              8:'Some other race alone', 9:'Two or more races'}
sex = group_summary(visible, ['SEX'])
sex['name'] = sex.SEX.map({1:'Male', 2:'Female'})
race = group_summary(visible, ['RAC1P'])
race['name'] = race.RAC1P.map(race_names)
display(sex)
display(race)
intersections = group_summary(visible, ['RAC1P', 'SEX'])
display(intersections.sort_values('n'))

,SEX,n,positives,positive_rate,negatives,name
0,1,118642,73087,0.616030,45555,Male
1,2,123470,64719,0.524168,58751,Female


,RAC1P,n,positives,positive_rate,negatives,name
0,1,151158,84961,0.562068,66197,White alone
1,2,12048,5766,0.478586,6282,Black alone
2,3,1788,876,0.489933,912,American Indian alone
3,4,17,6,0.352941,11,Alaska Native alone
4,5,575,297,0.516522,278,Other American Indian/Alaska Native category
5,6,39424,23557,0.597529,15867,Asian alone
6,7,774,432,0.558140,342,Native Hawaiian/Other Pacific Islander alone
7,8,26921,16306,0.605698,10615,Some other race alone
8,9,9407,5605,0.595833,3802,Two or more races


,RAC1P,SEX,n,positives,positive_rate,negatives
6,4,1,6,3,0.500000,3
7,4,2,11,3,0.272727,8
9,5,2,280,149,0.532143,131
8,5,1,295,148,0.501695,147
12,7,1,384,222,0.578125,162
13,7,2,390,210,0.538462,180
4,3,1,870,437,0.502299,433
5,3,2,918,439,0.478214,479
16,9,1,4663,2905,0.622989,1758
17,9,2,4744,2700,0.569140,2044


### Reading the group tables

`group_summary(...)` performs the same counting operation for each requested group. `.map(...)` adds readable names to numeric codes. Sorting intersections by `n` puts the smallest groups first.

| Column | Meaning |
| --- | --- |
| `n` | Number of sampled people in the group |
| `positives` | People with Y=1 |
| `negatives` | People with Y=0 |
| `positive_rate` | `positives / n` |

For example, this run has positive-label rates of about **61.6% for recorded males and 52.4% for recorded females** in non-test records. That is a difference in recorded outcomes, not a fairness result for a predictor. Later we will compare predictions and errors as well as true labels.

The race-by-sex table forms combinations of the two attributes. Some are very small: the Alaska Native-alone category has only 17 non-test records in total. A rate from a handful of people can change sharply with one record. Rows with `n=0` deliberately show absent groups; their positive rates are undefined.

## 6. Does each non-test split support evaluation?

EO needs actual positives and negatives in every calibration group. Small denominators make conditional rates unstable. The value 30 is a flag for discussion, not proof that larger groups are reliable.

**Inspect:** Which calibration groups have fewer than 30 examples of either label? We may need a clearly labeled supported-cohort analysis; we should not silently drop groups.

In [7]:
support = pd.read_csv(AUDIT / 'split_groups.csv')
support['small_label_count'] = support[['positives', 'negatives']].min(axis=1).lt(30)
display(support.loc[support['split'].eq('calibration')].sort_values(['small_label_count', 'n'], ascending=[False, True]))

,split,RAC1P,SEX,n,positives,positive_rate,negatives,small_label_count
6,calibration,4,1,0,0,NaN,0,True
7,calibration,4,2,1,1,1.000000,0,True
9,calibration,5,2,33,21,0.636364,12,True
8,calibration,5,1,37,16,0.432432,21,True
12,calibration,7,1,42,23,0.547619,19,True
13,calibration,7,2,47,24,0.510638,23,True
5,calibration,3,2,114,58,0.508772,56,False
4,calibration,3,1,122,61,0.500000,61,False
16,calibration,9,1,558,364,0.652330,194,False
17,calibration,9,2,583,327,0.560892,256,False


### What the support flag means

The support table breaks each split down by race and sex. The code selects the calibration rows, computes the smaller of `positives` and `negatives` for each group, and flags values below 30. It sorts flagged groups first, then by group size.

**True means review this group**, not that the data are corrupt or that it must be deleted. The threshold 30 is our discussion flag, not a statistical guarantee.

In this run, some race-by-sex calibration cells are empty or contain only one label. Without actual negatives, a group's false-positive rate is undefined; without actual positives, its true-positive rate is undefined. That prevents an ordinary equalized-odds fit over those intersectional groups. This is a support check for a possible intersectional analysis; our primary interventions still target sex and race separately. We must check those groupings too before fitting them.

## 7. What will the neural network receive?

We use 14 source input columns: scaled age plus one-hot categorical features. Sex and race stay in separate metadata. The scaler and category vocabulary are fitted only on the designated public reference subset. Unknown categories map to all-zero indicator blocks.

**Inspect:** Verify that ESR, Y, PWGTP, SEX, and RAC1P never appear in input columns. One-hot encoding increases the dimension beyond 14.

In [8]:
print('Source inputs:', INPUTS)
print('Encoded feature names:', manifest['encoded_features'])
for split, n in manifest['split_counts'].items():
    matrix = sparse.load_npz(DATA / f'X_{split}.npz')
    metadata = pd.read_csv(DATA / f'metadata_{split}.csv', usecols=['row_id'])
    assert matrix.shape == (n, manifest['encoded_dimension'])
    assert len(metadata) == n and metadata.row_id.is_unique
    assert np.isfinite(matrix.data).all()
    print(split, matrix.shape, matrix.dtype)
assert len(INPUTS) == 14
assert not set(['SEX', 'RAC1P', 'ESR', 'Y', 'PWGTP']) & set(INPUTS)
print('Alignment, dimensions, and finite-value checks passed.')

Source inputs: ['AGEP', 'SCHL', 'MAR', 'DIS', 'MIG', 'CIT', 'MIL', 'ANC', 'NATIVITY', 'RELP', 'DEAR', 'DEYE', 'DREM', 'GCL']
Encoded feature names: ['age__AGEP', 'categories__SCHL_1', 'categories__SCHL_10', 'categories__SCHL_11', 'categories__SCHL_12', 'categories__SCHL_13', 'categories__SCHL_14', 'categories__SCHL_15', 'categories__SCHL_16', 'categories__SCHL_17', 'categories__SCHL_18', 'categories__SCHL_19', 'categories__SCHL_2', 'categories__SCHL_20', 'categories__SCHL_21', 'categories__SCHL_22', 'categories__SCHL_23', 'categories__SCHL_24', 'categories__SCHL_3', 'categories__SCHL_4', 'categories__SCHL_5', 'categories__SCHL_6', 'categories__SCHL_7', 'categories__SCHL_8', 'categories__SCHL_9', 'categories__MAR_1', 'categories__MAR_2', 'categories__MAR_3', 'categories__MAR_4', 'categories__MAR_5', 'categories__DIS_1', 'categories__DIS_2', 'categories__MIG_1', 'categories__MIG_2', 'categories__MIG_3', 'categories__CIT_1', 'categories__CIT_2', 'categories__CIT_3', 'categories__CIT_4', '

### Why do 14 variables become 77 columns?

A categorical variable becomes several 0/1 indicator columns. For illustration, a variable with categories A, B, and C becomes three columns: A is `[1,0,0]`, B is `[0,1,0]`, and C is `[0,0,1]`. This is **one-hot encoding**. It avoids treating category codes as measurements with meaningful numerical distances.

Our saved representation has **1 scaled-age column + 76 indicator columns = 77 inputs per person**. Encoded names identify their origin: `categories__MAR_1`, for example, indicates membership in marital-status category 1. Age is centered/scaled using reference-set statistics; it is still one variable.

This final code cell is a file-consistency check. You can focus on the ordinary DataFrame sections first:

- `sparse.load_npz(...)` reads the numerical matrix for a split. Sparse storage saves space by storing mostly nonzero entries; it represents the same rectangular data idea as a table.
- `metadata` loads row identifiers to check row counts without displaying test labels.
- A shape such as `(151320, 77)` means **151,320 people × 77 numerical columns**.
- `float32` is the numerical storage type, not a performance score.
- The assertions check dimensions, unique identifiers, finite numerical values, and exclusion of target/protected columns from direct inputs.

**“Checks passed” means these preparation checks succeeded. It does not establish predictive accuracy or fairness.**

## 8. Your review notes

Edit this Markdown cell after inspecting the tables:

- Population/filter observations:
- Target definition and interpretation:
- Missingness rules to keep or revisit:
- Small groups or calibration-support concerns:
- Questions before the baseline model:

The next experiment is the non-private MLP. Keep the test set reserved while choosing model settings. Record any design changes in proposal.md before rerunning preparation.